In [1]:
import pandas as pd
import numpy as np

df_full = pd.read_parquet("data/processed/df_full.parquet")
df_flow = pd.read_parquet("data/processed/df_flow.parquet")

print(df_full.shape, df_flow.shape)

(2827761, 69) (2233050, 69)


In [1]:
def parse_ts(s):
    uniq = s.unique()
    parsed = pd.to_datetime(pd.Series(uniq), dayfirst=True, format="mixed", errors="coerce")
    return s.map(pd.Series(parsed.values, index=uniq))

for d in (df_full, df_flow):
    raw = d.groupby("day_file")["Timestamp"].transform(parse_ts)
    is_pm = raw.dt.hour < 8                                  # 1-7 o'clock = afternoon
    d["ts"] = raw + pd.to_timedelta(is_pm.astype(int) * 12, unit="h")

print(df_full["ts"].isna().sum(), df_flow["ts"].isna().sum())
print(df_full.groupby("day_file")["ts"].agg(["min", "max"]))

0 0
                                                                   min  \
day_file                                                                 
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv   2017-07-07 15:30:00   
Friday-WorkingHours-Afternoon-PortScan.pcap_ISC... 2017-07-07 13:00:00   
Friday-WorkingHours-Morning.pcap_ISCX.csv          2017-07-07 08:59:00   
Monday-WorkingHours.pcap_ISCX.csv                  2017-07-03 08:55:58   
Thursday-WorkingHours-Afternoon-Infilteration.p... 2017-07-06 13:00:00   
Thursday-WorkingHours-Morning-WebAttacks.pcap_I... 2017-07-06 08:59:00   
Tuesday-WorkingHours.pcap_ISCX.csv                 2017-07-04 08:53:00   
Wednesday-workingHours.pcap_ISCX.csv               2017-07-05 08:42:00   

                                                                   max  
day_file                                                                
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv   2017-07-07 17:02:00  
Friday-WorkingHours-Afternoon-PortSc

In [2]:
def assign_split(d):
   
    d = d.sort_values(["day_file", "ts"], kind="stable")

    g = d.groupby(["day_file", "Label"])
    frac = g.cumcount() / g["Label"].transform("size")

    d["split"] = np.select([frac < 0.6, frac < 0.8], ["train", "val"], default="test")
    return d.sort_index()   # put rows back in their original order

df_full = assign_split(df_full)
df_flow = assign_split(df_flow)

pd.crosstab(df_full["Label"], df_full["split"])[["train", "val", "test"]]

split,train,val,test
Label,,,
BENIGN,1362726,454240,454239
Bot,1174,391,391
DDoS,76815,25605,25605
DoS GoldenEye,6176,2059,2058
DoS Hulk,138075,46025,46024
DoS Slowhttptest,3300,1100,1099
DoS slowloris,3478,1159,1159
FTP-Patator,4761,1587,1587
Heartbleed,7,2,2


In [3]:
META = ["Flow ID", "Source IP", "Source Port", "Destination IP", "Destination Port",
        "Protocol", "Timestamp", "Label", "day_file", "ts", "split"]
FEATS = [c for c in df_full.columns if c not in META]
print(len(FEATS), "features")

for d in (df_full, df_flow):
    # 1. Init_Win: remember where it was missing, then set -1 to 0
    for col in ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]:
        d[col + "_missing"] = (d[col] < 0).astype("float32")
        d[col] = d[col].clip(lower=0)

    # 2. squash huge ranges
    d[FEATS] = np.log1p(d[FEATS].clip(lower=0)).astype("float32")

FEATS_ALL = FEATS + ["Init_Win_bytes_forward_missing", "Init_Win_bytes_backward_missing"]
print(len(FEATS_ALL), "features after adding the 2 flags")
df_full[FEATS_ALL].describe().T[["min", "max"]].head(10)

60 features
62 features after adding the 2 flags


,min,max
Flow Duration,0.693147,18.603003
Total Fwd Packets,0.693147,12.300291
Total Backward Packets,0.000000,12.584246
Total Length of Fwd Packets,0.000000,16.372738
Total Length of Bwd Packets,0.000000,20.300837
Fwd Packet Length Max,0.000000,10.119446
Fwd Packet Length Min,0.000000,7.751905
Fwd Packet Length Mean,0.000000,8.689777
Fwd Packet Length Std,0.000000,8.871589
Bwd Packet Length Max,0.000000,9.879758


In [4]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import joblib

train = df_full["split"] == "train"

# learn mean and std from the TRAIN rows only
scaler = StandardScaler().fit(df_full.loc[train, FEATS_ALL])

# apply the same scaling to everything
for d in (df_full, df_flow):
    d[FEATS_ALL] = scaler.transform(d[FEATS_ALL]).astype("float32")

# save the scaler so new data can be scaled the same way later (dashboard)
Path("models").mkdir(exist_ok=True)
joblib.dump(scaler, "models/scaler.joblib")

df_full.loc[train, FEATS_ALL].describe().T[["mean", "std"]].head(10)

,mean,std
Flow Duration,-1.381386e-08,1.0
Total Fwd Packets,-2.762771e-08,1.0
Total Backward Packets,5.295312e-08,1.0
Total Length of Fwd Packets,2.072078e-08,1.0
Total Length of Bwd Packets,9.209238e-09,1.0
Fwd Packet Length Max,2.993002e-08,1.0
Fwd Packet Length Min,8.058083e-09,1.0
Fwd Packet Length Mean,2.532540e-08,1.0
Fwd Packet Length Std,5.986004e-08,1.0
Bwd Packet Length Max,1.381386e-08,1.0


## Save checkpoint
Save the cleaned + scaled data so later notebooks don't need to redo the steps above.

In [6]:
import json

# full data: features + label/split + the columns needed to build per-IP sequences
full_cols = FEATS_ALL + ["Label", "split", "ts", "Source IP", "Destination IP", "Destination Port", "day_file"]
df_full[full_cols].to_parquet("data/processed/full_ready.parquet", index=False)

# flow data: only features + label + split (one row = one flow, no sequences)
df_flow[FEATS_ALL + ["Label", "split"]].to_parquet("data/processed/flow_ready.parquet", index=False)

# the feature list, so every notebook uses the same columns in the same order
with open("data/processed/features.json", "w") as f:
    json.dump(FEATS_ALL, f, indent=2)

print("full_ready:", df_full.shape[0], "rows x", len(full_cols), "cols")
print("flow_ready:", df_flow.shape[0], "rows x", len(FEATS_ALL) + 2, "cols")
print("features.json:", len(FEATS_ALL), "features")

full_ready: 2827761 rows x 69 cols
flow_ready: 2233050 rows x 64 cols
features.json: 62 features


## Per-IP sequence data (for the Transformer)
Idea: the Transformer looks at the last 32 flows sent by the same Source IP, not just one flow.
We store all flows sorted by (split, Source IP, time) in one big array, plus `grp_start` =
where each IP's block begins. A window for row `i` is then just rows `grp_start[i] .. i` (at most 32),
so we never have to build all windows in memory.

This part starts from `full_ready.parquet`, so it also works after a kernel restart.

In [7]:
import gc
import json
from pathlib import Path
import numpy as np
import pandas as pd

# free the big dataframes from above (they are saved to parquet now)
for name in ["df_full", "df_flow", "train"]:
    globals().pop(name, None)
gc.collect()

with open("data/processed/features.json") as f:
    FEATS_ALL = json.load(f)

cols = FEATS_ALL + ["Label", "split", "ts", "Source IP", "Destination IP", "Destination Port"]
seq = pd.read_parquet("data/processed/full_ready.parquet", columns=cols)

# put each IP's flows next to each other, in time order, separately for train / val / test.
# "stable" keeps the original file order for flows with the same timestamp (same minute).
seq = seq.sort_values(["split", "Source IP", "ts"], kind="stable").reset_index(drop=True)
print(seq.shape)

(2827761, 68)


In [8]:
# True if this row belongs to the same (split, Source IP) group as the row above it
same_grp = (seq["split"] == seq["split"].shift()) & (seq["Source IP"] == seq["Source IP"].shift())

# behaviour features: did this source talk to a NEW destination IP / port compared to its previous flow?
# the first flow of a group has no previous flow, so it counts as new (= 1)
seq["new_dst_ip"] = (~same_grp | (seq["Destination IP"] != seq["Destination IP"].shift())).astype("float32")
seq["new_dst_port"] = (~same_grp | (seq["Destination Port"] != seq["Destination Port"].shift())).astype("float32")

SEQ_FEATS = FEATS_ALL + ["new_dst_ip", "new_dst_port"]
print(len(SEQ_FEATS), "sequence features")

# grp_start: row index where each row's group begins.
# group starts keep their own index, other rows get 0, then a running max copies the start downwards.
row = np.arange(len(seq), dtype=np.int64)
grp_start = np.maximum.accumulate(np.where(same_grp, 0, row))

64 sequence features


In [9]:
LABELS = sorted(seq["Label"].unique())
SPLIT_ID = {"train": 0, "val": 1, "test": 2}

X = seq[SEQ_FEATS].to_numpy(dtype=np.float32)                                     # rows x 64
y = pd.Categorical(seq["Label"], categories=LABELS).codes.astype(np.int8)          # label id = position in LABELS
split = seq["split"].map(SPLIT_ID).to_numpy(dtype=np.int8)                         # 0 / 1 / 2

out = Path("data/seq")
out.mkdir(parents=True, exist_ok=True)
np.save(out / "X.npy", X)
np.save(out / "y.npy", y)
np.save(out / "split.npy", split)
np.save(out / "grp_start.npy", grp_start)
with open(out / "meta.json", "w") as f:
    json.dump({"features": SEQ_FEATS, "labels": LABELS}, f, indent=2)

print("saved to", out, "| X", X.shape, X.dtype, "| size %.0f MB" % (X.nbytes / 1e6))

# the dataframe is not needed any more
del seq, same_grp, row, X, y, split, grp_start
_ = gc.collect()   # "_ =" stops the notebook from printing the returned number

saved to data\seq | X (2827761, 64) float32 | size 724 MB


## Sanity checks
Load the saved arrays back (memory-mapped = read from disk on demand) and check them.

In [10]:
X = np.load("data/seq/X.npy", mmap_mode="r")
y = np.load("data/seq/y.npy")
split = np.load("data/seq/split.npy")
grp_start = np.load("data/seq/grp_start.npy")
meta = json.load(open("data/seq/meta.json"))
LABELS, SEQ_FEATS = meta["labels"], meta["features"]

print("X", X.shape, X.dtype, "| y", y.shape, y.dtype, "| split", split.shape, split.dtype,
      "| grp_start", grp_start.shape, grp_start.dtype)

n_groups = (grp_start == np.arange(len(grp_start))).sum()
print("number of (split, Source IP) groups:", n_groups)

X (2827761, 64) float32 | y (2827761,) int8 | split (2827761,) int8 | grp_start (2827761,) int64
number of (split, Source IP) groups: 29693


In [11]:
# PortScan hits many ports one after another, so new_dst_port should be close to 1
port_col = SEQ_FEATS.index("new_dst_port")
ip_col = SEQ_FEATS.index("new_dst_ip")
for lab in ["PortScan", "BENIGN"]:
    rows = y == LABELS.index(lab)
    print(f"{lab:9s} mean new_dst_port = {X[rows, port_col].mean():.3f}   mean new_dst_ip = {X[rows, ip_col].mean():.3f}")

PortScan  mean new_dst_port = 0.999   mean new_dst_ip = 0.000
BENIGN    mean new_dst_port = 0.353   mean new_dst_ip = 0.343


In [12]:
# label counts per split, plus each split's share of the label (should be about 60 / 20 / 20 %)
SPLIT_NAMES = ["train", "val", "test"]
counts = pd.crosstab(pd.Series(np.array(LABELS)[y], name="Label"),
                     pd.Series(np.array(SPLIT_NAMES)[split], name="split"))[SPLIT_NAMES]
share = (counts.div(counts.sum(axis=1), axis=0) * 100).round(1).add_suffix(" %")
pd.concat([counts, share], axis=1)

split,train,val,test,train %,val %,test %
Label,,,,,,
BENIGN,1362726,454240,454239,60.0,20.0,20.0
Bot,1174,391,391,60.0,20.0,20.0
DDoS,76815,25605,25605,60.0,20.0,20.0
DoS GoldenEye,6176,2059,2058,60.0,20.0,20.0
DoS Hulk,138075,46025,46024,60.0,20.0,20.0
DoS Slowhttptest,3300,1100,1099,60.0,20.0,20.0
DoS slowloris,3478,1159,1159,60.0,20.0,20.0
FTP-Patator,4761,1587,1587,60.0,20.0,20.0
Heartbleed,7,2,2,63.6,18.2,18.2


In [13]:
# a group must never mix train / val / test rows (that would leak information between splits)
ok = (split == split[grp_start]).all()
print("no group crosses splits:", ok)
assert ok

no group crosses splits: True


In [14]:
def get_window(i, L=32):
    """Last L flows of row i's source IP (ending at row i), left-padded with zeros.
    mask is True for real flows and False for padding."""
    start = max(grp_start[i], i - L + 1)        # don't go back past the start of the group
    rows = X[start:i + 1]
    n = len(rows)
    win = np.zeros((L, X.shape[1]), dtype=np.float32)
    mask = np.zeros(L, dtype=bool)
    win[L - n:] = rows                          # real flows go at the end
    mask[L - n:] = True
    return win, mask

for lab in ["PortScan", "BENIGN"]:
    i = np.flatnonzero(y == LABELS.index(lab))[1000]    # just some row with this label
    win, mask = get_window(i)
    print(f"{lab:9s} row {i}: window {win.shape}, mask {mask.shape}, real flows = {mask.sum()}")

PortScan  row 70051: window (32, 64), mask (32,), real flows = 32
BENIGN    row 1000: window (32, 64), mask (32,), real flows = 4


## Preprocessing summary

**Files saved**
- `models/scaler.joblib`: the StandardScaler (fitted on train rows only). Use it to scale new data the same way.
- `data/processed/features.json`: the 62 feature names (`FEATS_ALL`), in order.
- `data/processed/full_ready.parquet`: all flows. 62 scaled features + Label, split, ts, Source IP, Destination IP, Destination Port, day_file.
- `data/processed/flow_ready.parquet`: deduplicated flows. 62 scaled features + Label, split.
- `data/seq/X.npy`: float32, rows x 64. The 62 features + `new_dst_ip` + `new_dst_port`, sorted by (split, Source IP, time).
- `data/seq/y.npy`: int8 label id for each row (position in `labels` in meta.json).
- `data/seq/split.npy`: int8. 0 = train, 1 = val, 2 = test.
- `data/seq/grp_start.npy`: int64. The row where each row's (split, Source IP) group begins.
- `data/seq/meta.json`: `features` (64 names) and `labels` (sorted label list).

**What was done**
- Fixed timestamps (12-hour clock) into `ts`.
- Split 60 / 20 / 20 by time inside each (day file, label) group.
- Added 2 "missing" flags for Init_Win, clipped -1 to 0, applied log1p, then scaled.
- For sequences: added 2 behaviour features and checked that no IP group mixes splits.

**Next notebook (03_models) will load**
- `flow_ready.parquet` + `features.json` for the single-flow models (train on split == "train").
- `data/seq/*.npy` + `meta.json` for the Transformer, cutting 32-flow windows on the fly with `get_window`.
- `models/scaler.joblib` only when new raw data needs scaling.